# EDEF Linear Classification Quickstart

EDEF decomposes realized classification fit, not predictions.

For binary classification, EDEF explains reductions in realized log loss.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

import edef


In [ ]:
rng = np.random.default_rng(123)

X = rng.normal(size=(500, 3))

beta = np.array([1.5, 0.75, 0.0])
eta = X @ beta - 0.25
p = 1.0 / (1.0 + np.exp(-eta))

y = rng.binomial(1, p, size=500)

feature_names = ["x1_signal", "x2_signal", "x3_noise"]


In [ ]:
model = LogisticRegression(C=1e6, solver="lbfgs")
model.fit(X, y)

explainer = edef.LinearExplainer(
    model,
    loss="log_loss",
    feature_names=feature_names,
)

result = explainer(X, y)


In [ ]:
print(result)


In [ ]:
result.to_frame()


In [ ]:
ax = result.plot()
ax.set_title("EDEF contributions to realized binary log-loss fit")
plt.show()


The `__InterceptShift__` term appears because the fitted model intercept generally differs from the best constant baseline logit.

The feature contributions and intercept-shift contribution add to the realized log-loss improvement.


In [ ]:
grouped = result.group(
    ["signal", "signal", "noise", "intercept"]
)

print(grouped)


In [ ]:
ax = grouped.plot()
ax.set_title("Grouped EDEF contributions")
plt.show()


Unlike prediction attribution, EDEF depends on the realized labels `y`.

A feature can have large prediction effects but small or negative fit contributions if those prediction effects do not improve realized log loss.


## SHAP Plotting Compatibility


In [ ]:
shap_exp = result.to_shap_explanation(data=X)


In [ ]:
import shap

shap.plots.beeswarm(shap_exp)
